In [1]:
from utility.plot_spectrum import plot_spectrum
from utility.adjust_redshift import adjust_redshift

from processing.remove_spikes import remove_spikes
from processing.fit_continuum import fit_continuum
from processing.quality_cuts import quality_cuts
from processing.make_bal_mask import make_bal_mask
from processing.fit_gaussians import fit_single_gaussian, fit_two_gaussians, ftest_which_gaussian
from processing.measure_line import measure_line
from processing.parameters import CIV_CONTINUUM_WINDOWS

from sparcl.client import SparclClient
from dl import queryClient as qc
from astropy.table import Table
from astropy.convolution import convolve, Gaussian1DKernel
from scipy.interpolate import interp1d
from datetime import datetime
from scipy.signal import argrelmin, argrelmax

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
import json
import os

hamann_fits_path = "data/local/ERQs_core_all_MNRAS.fits"
hamann_coords_path = "data/core-ERQ-stats-coords.csv"
search_log_path = "data/local/hamann-objects-sdss-search-log.json"
hamann_sdss_spectra_path = "data/local/hamann_sdss_spectra.json"
relevant_erq_data_path = "data/relevant_erq_data.csv"


# client = SparclClient()

In [6]:
client = SparclClient()

announcement=Data set deprecation notice: on November 19, 2025 the SDSS/BOSS DR16 data sets were deprecated. Please use the new SDSS/BOSS DR17 data sets instead.


### find spectra

In [7]:
hamann_coords = pd.read_csv(hamann_coords_path, index_col=0)

hamann_coords

,SDSS_id,RA,DEC,ze,i,i-W3,REW,REW_unc,FWHM,FWHM_unc,kt80,NV/CIV,BAL,UV_slope,E(B-V),E(B-V)_unc,FIRST
0,J000610.67+121501.2,1.544491,12.250354,2.31,22.1,8.0,107.0,6.0,4540.0,200.0,0.37,1.88,0,-0.20,0.66,0.01,0.0
1,J000746.19+122223.9,1.942491,12.373315,2.43,21.3,4.9,220.0,6.0,2433.0,71.0,0.28,0.39,0,-0.17,0.19,0.01,0.0
2,J002400.25+245031.9,6.001042,24.842208,2.81,21.6,6.0,144.0,10.0,3523.0,195.0,0.37,2.46,0,-1.14,0.20,0.02,NaN
3,J004713.21+264024.7,11.805059,26.673525,2.56,20.4,4.7,101.0,2.0,3685.0,74.0,0.33,1.92,1,-1.08,0.15,0.01,NaN
4,J005044.95-021217.6,12.687326,-2.204915,2.25,21.5,4.7,147.0,9.0,4343.0,487.0,0.19,0.61,0,-1.73,0.14,0.02,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,J223754.52+065026.6,339.477177,6.840747,2.61,22.0,5.8,141.0,5.0,1391.0,44.0,0.36,1.03,0,-1.20,0.24,0.02,0.0
93,J225438.30+232714.5,343.659621,23.454052,3.09,22.0,5.5,415.0,20.0,4412.0,146.0,0.36,0.66,0,-0.38,NaN,NaN,NaN
94,J232326.17-010033.1,350.859062,-1.009201,2.36,22.4,7.2,256.0,5.0,3989.0,62.0,0.37,2.16,0,0.91,0.40,0.01,0.0
95,J232828.47+044346.8,352.118656,4.729677,2.56,21.5,5.5,359.0,5.0,1584.0,20.0,0.35,0.37,0,-0.28,0.13,0.02,0.0


In [8]:
def log_result_to_json(sdss_id,best_match_id,results_df, filepath=search_log_path):
    '''Adds best match log entry to specified JSON file.'''

    # create entry
    entry = {
        "timestamp":     datetime.utcnow().isoformat(),
        "SDSS_id":       sdss_id,
        "best_match_id": best_match_id,
        "results_df":    results_df.to_dict(orient="records")
    }

    # open file
    with open(filepath, "r") as f:
        log = json.load(f)

    log.append(entry)  # append entry

    # save to file
    with open(filepath, "w") as f:
        json.dump(log, f, indent=2, default=str)
    

    # if accept_two:
    #     line_stats = measure_line(lam, profile2, flux_sub, continuum)
    #     plot_spectrum(lam[plot_mask], flux_sub[plot_mask], flux_label="continuum subtracted flux", add_smoothed=False, additional_flux=profile2[plot_mask], additional_flux_label="line fit", line_lambda=[1549], line_label=["CIV"], line_width_factor=2)
    # else:
    #     line_stats = measure_line(lam, profile1, flux_sub, continuum)
    #     plot_spectrum(lam[plot_mask], flux_sub[plot_mask], flux_label="continuum subtracted flux", add_smoothed=False, additional_flux=profile1[plot_mask], additional_flux_label="line fit", line_lambda=[1549], line_label=["CIV"], line_width_factor=2)
    print(f"Logged entry for {sdss_id}")

In [ ]:
def query_sdss_by_coords(df, z_deviation=0.5, coord_deviaion=0.1):
    '''
    Iterates through dataframe, for each element searches in DESI for the object with smallest difference in RA, DEC, and z (in that order of importance).
    '''

    found_idx = []

    with open(search_log_path, "r") as f:
        log = json.load(f)
        for elem in log:
            found_idx.append(elem["SDSS_id"])

    for elem_idx in range(len(df)):
        # extract ra, dec, z and id for current element
        elem_ra = df["RA"][elem_idx]
        elem_dec = df["DEC"][elem_idx]
        elem_z = df["ze"][elem_idx]
        elem_sdss_id = df["SDSS_id"][elem_idx]

        if elem_sdss_id not in found_idx:

            # query for objects within set devations of current element 
            constraints = {
                "data_release":["BOSS-DR17"],
                "ra":[elem_ra-coord_deviaion,elem_ra+coord_deviaion],
                "dec":[elem_dec-coord_deviaion,elem_dec+coord_deviaion],
                "redshift":[elem_z-z_deviation,elem_z+z_deviation]
                }

            results = client.find(outfields=["specid","ra","dec","redshift"], constraints=constraints)

            results = pd.DataFrame(results.data[1:])

            # if there are results, calculate differences, then sort to get best match
            if len(results) > 0:
                results["z_diff"] = abs(results["redshift"] - elem_z)
                results["ra_diff"] = abs(results["ra"] - elem_ra)
                results["dec_diff"] = abs(results["dec"] - elem_dec)
                results["coord_diff"] = (results["ra"] - elem_ra)**2 + (results["dec"] - elem_dec)**2
                results = results.sort_values(["coord_diff","z_diff"], ascending=[True,True])

                best_match = results["specid"][0]

            # if not found, say so
            else:
                best_match = "Not found"            


            # log to json file
            log_result_to_json(elem_sdss_id,best_match,results)

In [19]:
found_idx = []

with open(search_log_path, "r") as f:
        log = json.load(f)
        # print(log)
        for elem in log:
            # print(elem)
            found_idx.append(elem["SDSS_id"])

In [20]:
len(set(found_idx))

97

#### <span style="color:red">something about this does not work I did not bother to check what, like the "best match" is  not correct sometimes</span>

In [18]:
query_sdss_by_coords(hamann_coords)

                specid         ra  redshift       dec        _dr    z_diff  \
2  4740064262789289984  350.85906  2.350765 -1.009201  BOSS-DR17  0.009235   
0  4741289393654093824  350.93459  2.106846 -1.046688  BOSS-DR17  0.253154   
1  4740050793771849728  350.93459  2.104943 -1.046688  BOSS-DR17  0.255057   
3 -8139068444095784960  350.76164  1.998726 -1.074235  BOSS-DR17  0.361274   
4 -8139066519950436352  350.75924  2.199740 -0.918895  BOSS-DR17  0.160260   

    ra_diff      dec_diff    coord_diff  
2  0.000002  3.000000e-07  4.090000e-12  
0  0.075528  3.748680e-02  7.109739e-03  
1  0.075528  3.748680e-02  7.109739e-03  
3  0.097422  6.503360e-02  1.372042e-02  
4  0.099822  9.030566e-02  1.811954e-02  
Logged entry for J232326.17-010033.1
                specid         ra  redshift       dec        _dr    z_diff  \
0  4822392127110797312  352.11866  2.568358  4.729677  BOSS-DR17  0.008358   
1  4822393226622425088  352.06002  2.218850  4.662951  BOSS-DR17  0.341150   

    ra_

In [21]:
hamann_data = Table.read("data/local/ERQs_core_all_MNRAS.fits").to_pandas() #pd.read_csv(hamann_erq_path,index_col=0)

hamann_data["sdss_name"] = hamann_data["sdss_name"].astype("string")

match_df = pd.read_json(search_log_path)
match_df = match_df[["SDSS_id","best_match_id"]].copy()
match_df["SDSS_id"] = match_df["SDSS_id"].str[1:]
match_df["best_match_id"] = match_df["best_match_id"].astype("string")

match_df = match_df.rename(columns={"best_match_id":"specid", "SDSS_id":"sdss_name"})

hamann_erq = hamann_data.merge(match_df,on="sdss_name",how="left")

hamann_erq = hamann_erq.drop_duplicates()
# hamann_erq = match_df.merge(hamann_data,on="sdss_name",how="right")
hamann_erq

,sdss_name,Plate,MJD,FiberID,ThingID,z_dr12,i,i_err,W3,W3_snr,...,kt75,kt80,rew_di,rewe_di,fwhm_di,qflag,rew_nv,frat_nv_civ,nvflag,specid
0,000610.67+121501.2,5649,55912,288,220079389,2.309943,22.101451,0.160805,14.087,35.1,...,0.455485,0.372288,102.007661,6.075000,4508.508813,0,192.343862,1.880187,0,-5430021594299475968
1,000746.19+122223.9,6176,56264,194,221814072,2.429000,21.314343,0.079014,16.369,7.0,...,0.383122,0.284162,213.628344,4.754235,2550.825969,0,64.601598,0.388502,0,6360267496814303232
2,002400.25+245031.9,6279,56243,28,328146826,2.809255,21.556824,0.090767,15.522,14.6,...,0.455389,0.372201,134.821509,10.120908,3694.810492,0,205.776721,2.455813,0,7069530567608653824
3,004713.21+264024.7,6286,56301,800,342571394,2.563943,20.435463,0.038392,15.714,13.2,...,0.413071,0.325082,100.642397,5.476974,3662.920844,0,141.614347,1.923201,0,7077626822452860928
4,005044.95-021217.6,4370,55534,980,60738765,2.254000,21.473060,0.078552,16.760,4.6,...,0.262998,0.193256,148.782256,6.993491,5445.426756,0,46.553849,0.609687,0,4920452066097518592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,223754.52+065026.6,5058,56208,124,172688138,2.608809,21.960453,0.106963,16.169,3.9,...,0.440905,0.356129,136.337135,3.430820,1318.560329,0,109.072962,1.034336,0,5694835917824743424
93,225438.30+232714.5,6588,56536,390,315135318,3.087494,22.010175,0.120658,16.552,5.3,...,0.442837,0.358143,354.839828,22.560964,4906.586551,0,211.097222,0.660819,0,7417523253936412672
94,232326.17-010033.1,4210,55444,93,75229201,2.356070,22.410147,0.200206,15.218,19.1,...,0.454002,0.370693,260.264938,4.465699,4186.298391,0,565.970265,2.160131,0,4741289393654093824
95,232828.47+044346.8,4283,55864,592,153323514,2.563872,21.532625,0.144071,16.066,9.3,...,0.437510,0.352179,356.552346,4.328203,1575.355245,0,122.280344,0.373535,0,4822392127110797312


In [24]:
idxs = [int(x) for x in hamann_erq["specid"] if x != "Not found"]

output_fields = ['specid', 'redshift', 'wavelength', 'flux', 'ivar', 'spectype', 'redshift_warning']

spectra = client.retrieve_by_specid(idxs, include=output_fields)

In [26]:
spectra_df = pd.DataFrame(spectra.data[1:])

spectra_df.to_json(hamann_sdss_spectra_path,default_handler=str)

In [28]:
spectra_df

,redshift_warning,sparcl_id,specid,spectype,redshift,ivar,flux,wavelength,_dr
0,0,012e8ff6-2501-11f0-9d1c-525400f334e1,7462679094125418496,QSO,3.035928,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[9.9496011734, 9.9471158981, 9.9446353912, 9.9...","[3565.3321878293, 3566.1532304264, 3566.974462...",BOSS-DR17
1,0,0308562e-2497-11f0-8fd2-525400f334e1,6638542347197241344,QSO,2.414080,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[2.8909749985, 2.8905422688, 2.8901104927, 2.8...","[3562.8701940455, 3563.6906696824, 3564.511334...",BOSS-DR17
2,0,03e07120-24aa-11f0-af3b-525400f334e1,6778028042235435008,QSO,2.393440,"[0.039093356600000004, 0.0734360963, 0.0861192...","[-24.6522712708, 23.0006732941, -13.3715238571...","[3584.2640804371, 3585.0894827655, 3585.915075...",BOSS-DR17
3,0,07a9ddae-251c-11f0-8d2d-525400f334e1,7611298432574314496,QSO,2.332547,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.9814393520000002, 1.9814147949, 1.981390595...","[3565.3321878293, 3566.1532304264, 3566.974462...",BOSS-DR17
4,0,0840139a-24c7-11f0-943f-525400f334e1,4804230387667851264,QSO,3.137673,"[0.0, 0.1845502406, 0.0, 0.25157165530000003, ...","[-0.6954147220000001, -0.6955951452, -0.986162...","[3565.3321878293, 3566.1532304264, 3566.974462...",BOSS-DR17
...,...,...,...,...,...,...,...,...,...
92,64,f04676c4-2521-11f0-a4f5-525400f334e1,7657261318190422016,QSO,3.297671,"[0.0172611196, 0.053034059700000004, 0.0462748...","[0.7039064169, -1.2401340008, 0.88244611020000...","[3576.020501809, 3576.8440057648, 3577.6676993...",BOSS-DR17
93,0,f35301dd-24f0-11f0-a354-525400f334e1,7307270271760881664,QSO,2.407903,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[8.3524827957, 8.3522090912, 8.351937294, 8.35...","[3563.6906696824, 3564.5113342624, 3565.332187...",BOSS-DR17
94,4,f7891f2f-2569-11f0-8daa-525400f334e1,5985400552917587968,QSO,2.312891,"[0.0, 0.15839064120000002, 0.0, 0.1791924983, ...","[6.7276959419, 6.7282114028999995, 3.888189077...","[3553.0392081527, 3553.8574198608, 3554.675819...",BOSS-DR17
95,0,fbde5abc-2523-11f0-9745-525400f334e1,7667642906308794368,QSO,2.927785,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[9.5618829727, 9.5657606125, 9.5696353912, 9.5...","[3570.2612803902, 3571.0834580835, 3571.905825...",BOSS-DR17


### make clean df

In [29]:
spectra_df = pd.read_json(hamann_sdss_spectra_path)

spectra_df["specid"] = spectra_df["specid"].astype("string")

data = hamann_erq.merge(spectra_df,on="specid")

cols_to_keep = ['sdss_name', 'z_dr12', 'rew', 'fwhm', 'kt80', 'specid',
       'redshift', 'redshift_warning', 'wavelength', 'flux', 'ivar']
data = data[cols_to_keep].copy()

data

,sdss_name,z_dr12,rew,fwhm,kt80,specid,redshift,redshift_warning,wavelength,flux,ivar
0,000610.67+121501.2,2.309943,106.996659,4540.061012,0.372288,-5430021594299475968,2.195029,0,"[3585.0894827655, 3585.9150751717, 3586.740857...","[-3.3389332294000003, -3.3395802975, 0.6752824...","[0.0, 0.054592851500000004, 0.0, 0.0723279193,..."
1,000746.19+122223.9,2.429000,219.633801,2432.830081,0.284162,6360267496814303232,2.433351,0,"[3553.8574198608, 3554.6758199907, 3555.494408...","[-5.924967289, -5.9247941971, 3.1096043587, 12...","[0.0, 0.11920769510000001, 0.0, 0.0926503986, ..."
2,002400.25+245031.9,2.809255,143.689957,3523.281699,0.372201,7069530567608653824,2.652466,0,"[3558.7706484879, 3559.5901800608, 3560.409900...","[7.289812088, 7.2911128998, 5.6182889938, 3.94...","[0.0, 0.0583016686, 0.0, 0.0595223345, 0.06357..."
3,004713.21+264024.7,2.563943,100.506344,3685.427543,0.325082,7077626822452860928,2.553878,0,"[3561.2298094276, 3562.0499073084, 3562.870194...","[-1.7795865536000002, -1.7798964977, -1.780206...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,005044.95-021217.6,2.254000,147.225015,4343.057592,0.193256,4920452066097518592,2.253475,0,"[3553.0392081527, 3553.8574198608, 3554.675819...","[1.8903125525000002, 1.8905686140000002, 1.890...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...,...,...,...,...,...,...,...
92,223754.52+065026.6,2.608809,140.676812,1390.574742,0.356129,5694835917824743424,2.614897,0,"[3561.2298094276, 3562.0499073084, 3562.870194...","[14.607211113, -22.7902507782, 20.6671199799, ...","[0.09246767310000001, 0.1343497932, 0.12748730..."
93,225438.30+232714.5,3.087494,415.244897,4411.577590,0.358143,7417523253936412672,3.438205,0,"[3568.6174928348, 3569.4392919885, 3570.261280...","[-2.2476668358, 1.1492393017, -1.9651819468, -...","[0.1280496418, 0.29582172630000003, 0.32419168..."
94,232326.17-010033.1,2.356070,255.993179,3989.269919,0.370693,4741289393654093824,2.106846,0,"[3564.5113342624, 3565.3321878293, 3566.153230...","[8.7531776428, -2.8003044128, -4.5461950302, 4...","[0.0339920819, 0.1078312397, 0.132573619500000..."
95,232828.47+044346.8,2.563872,358.595759,1583.693351,0.352179,4822392127110797312,2.568358,0,"[3553.0392081527, 3553.8574198608, 3554.675819...","[1.856464982, 1.8564176559, 1.8563705683, 1.85...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [ ]:
def get_absorption_interval(lam, flux, ivar):
    mask = (lam > 1450) & (lam < 1650)

    lam, flux, ivar = lam[mask], flux[mask], ivar[mask]
    flux_conv = convolve(flux, Gaussian1DKernel(3))

    noise = np.where(ivar > 0, 1.0 / np.sqrt(np.maximum(ivar, 1e-30)), np.inf)

    mins = argrelmin(flux_conv)
    maxs = argrelmax(flux_conv)
    extrema_idxs = sorted(list(mins[0])+list((maxs[0])))
    mins_list = [(lam[i],flux_conv[i],noise[i],0) for i in mins[0]]
    maxs_list = [(lam[i],flux_conv[i],noise[i],1) for i in maxs[0]]
    extrema = pd.DataFrame(sorted(mins_list + maxs_list),columns=["lam","flux_conv","noise","max_bool"])
    extrema["flux_diff_backward"] = extrema["flux_conv"].diff().abs()
    extrema["flux_diff_forward"] = extrema["flux_conv"].diff(-1).abs()
    extrema["more_than_noise"] = (extrema["flux_diff_backward"] > 1.5*extrema["noise"]) & (extrema["flux_diff_forward"] > 1.5*extrema["noise"])
    extrema["overall_max_bool"] = extrema["flux_conv"] == extrema["flux_conv"].max()

    query = extrema.query("more_than_noise == 1 and max_bool == 0")

    if len(query) > 0:
        idx = query.index
        try:
            if extrema["overall_max_bool"][idx-1].squeeze() == True or extrema["overall_max_bool"][idx+1].squeeze() == True:

                lower = extrema["lam"][idx-1].squeeze()
                upper = extrema["lam"][idx+1].squeeze()
                five_percent = (upper - lower) * 0.05  # um nen puffer zu haben quasi

                absorption_interval = (lower+five_percent, upper-five_percent)

                return absorption_interval

        except ValueError:
            print("ValueError in get_absorption_interval()")

    return None

In [ ]:
def check(lam,flux,ivar):
    flux_clean, spike_mask = remove_spikes(lam,flux,ivar=ivar,sigma_thresh=3)

    result, continuum, window_mask = fit_continuum(lam, flux_clean, ivar=ivar)

    rejected, reason, flags = quality_cuts(lam, flux_clean, continuum, result, max_slope=10)
    
    if rejected:
        print(f'SDSS: {hamann_stats_for_this_idx["sdss_name"].squeeze()}\nRejected because: {reason}')
        plot_spectrum(lam, flux_clean, add_smoothed=False)
        return 

    flux_sub = flux_clean - continuum if not rejected else None

    absorption_interval = get_absorption_interval(lam,flux,ivar)
    
    if absorption_interval is not None:
        bal_mask = (lam > absorption_interval[0]) & (lam < absorption_interval[1])
    else:
        bal_mask = None
    # bal_mask = make_bal_mask(lam, flux_sub, ivar=ivar, min_width_aa=5)

    result1, profile1 = fit_single_gaussian(lam, flux_sub, ivar=ivar, mask=bal_mask)
    result2, profile2 = fit_two_gaussians(lam, flux_sub, ivar=ivar, mask=bal_mask, single_result=result1)

    accept_two, p_val = ftest_which_gaussian(result1, result2)

    if accept_two:
        sigma_w = result2.params["sigma_w"].value
        sigma_c = result2.params["sigma_c"].value
        mu_w = result2.params["cen_w"].value
        mu_c = result2.params["cen_c"].value

        lower = min(mu_c-3*sigma_c,mu_w-3*sigma_w)
        upper = max(mu_c+3*sigma_c,mu_w+3*sigma_w)
        ci = (lower,upper)

    else:
        mu = result1.params["center"].value
        sigma = result1.params["sigma"].value
        ci = (mu-3*sigma, mu+3*sigma)


    plot_mask = (lam >= 1400) & (lam <= 1700)
    continuum_mask = plot_mask#(lam >= 1425) & (lam <= 1810)

    if absorption_interval is not None:
        if accept_two:
            line_stats, win = measure_line(lam, profile2, flux_sub, continuum, ci, get_window=True)
            plot_spectrum(lam[plot_mask], flux[plot_mask], flux_label="flux",
                line_width_factor=2,
                add_smoothed=False,
                additional_lam=[lam[continuum_mask]]*2,
                additional_flux=[profile2[continuum_mask]+continuum[continuum_mask],continuum[continuum_mask]],
                additional_flux_label=["line fit","continuum fit"],
                line_lambda=[1549],
                line_label=["CIV"],
                shaded_regions=CONTINUUM_WINDOWS[:2]+[(lam[win][0], lam[win][-1]), absorption_interval],
                shaded_regions_colors=["green"]*2+["orange","DodgerBlue"],
                shaded_regions_label=["continuum fit"]*2+["integration","absorption mask"]
                )
        else:
            line_stats, win = measure_line(lam, profile1, flux_sub, continuum, ci, get_window=True)
            plot_spectrum(lam[plot_mask], flux[plot_mask], flux_label="flux",
                line_width_factor=2,
                add_smoothed=False,
                additional_lam=[lam[continuum_mask]]*2,
                additional_flux=[profile1[continuum_mask]+continuum[continuum_mask], continuum[continuum_mask]],
                additional_flux_label=["line fit","continuum fit"],
                line_lambda=[1549],
                line_label=["CIV"],
                shaded_regions=CONTINUUM_WINDOWS[:2]+[(lam[win][0], lam[win][-1]), absorption_interval],
                shaded_regions_colors=["green"]*2+["orange","DodgerBlue"],
                shaded_regions_label=["continuum fit"]*2+["integration","absorption mask"]
                )
        return True, line_stats
    else:
        if accept_two:
            line_stats, win = measure_line(lam, profile2, flux_sub, continuum, ci, get_window=True)
            plot_spectrum(lam[plot_mask], flux[plot_mask], flux_label="flux",
                line_width_factor=2,
                add_smoothed=False,
                additional_lam=[lam[continuum_mask]]*2,
                additional_flux=[profile2[continuum_mask]+continuum[continuum_mask],continuum[continuum_mask]],
                additional_flux_label=["line fit","continuum fit"],
                line_lambda=[1549],
                line_label=["CIV"],
                shaded_regions=CONTINUUM_WINDOWS[:2]+[(lam[win][0], lam[win][-1])],
                shaded_regions_colors=["green"]*2+["orange"],
                shaded_regions_label=["continuum fit"]*2+["EW integration"]
                )
        else:
            line_stats, win = measure_line(lam, profile1, flux_sub, continuum, ci, get_window=True)
            plot_spectrum(lam[plot_mask], flux[plot_mask], flux_label="flux",
                line_width_factor=2,
                add_smoothed=False,
                additional_lam=[lam[continuum_mask]]*2,
                additional_flux=[profile1[continuum_mask]+continuum[continuum_mask], continuum[continuum_mask]],
                additional_flux_label=["line fit","continuum fit"],
                line_lambda=[1549],
                line_label=["CIV"],
                shaded_regions=CONTINUUM_WINDOWS[:2]+[(lam[win][0], lam[win][-1])],
                shaded_regions_colors=["green"]*2+["orange"],
                shaded_regions_label=["continuum fit"]*2+["EW integration"]
                )
        return False, line_stats
        

In [ ]:
# data["absorption_detected"] = None
# data["absorption_relevant"] = None

# pd.options.mode.chained_assignment = None

In [ ]:
# for i in range(len(data)):
#     lam = np.array(data["wavelength"][i])
#     flux = np.array(data["flux"][i])
#     ivar = np.array(data["ivar"][i])
#     z = data["redshift"][i]

#     lam, flux, ivar = adjust_redshift(z, lam, flux, ivar=ivar)

#     absorption_found, line_stats = check(lam, flux, ivar)

#     out_str = f'''
#                 SDSS: {data["sdss_name"][i]}
#                 FWHM: {round(line_stats["fwhm_kms"],3)} vs {round(data["fwhm"][i],3)} (Hamann)
#                 REW: {round(line_stats["ew_aa"],3)} vs {round(data["rew"][i],3)} (Hamann)
#                 kt80: {round(line_stats["kt80"],3)} vs {round(data["kt80"][i],3)} (Hamann)
#     '''
#     print(out_str)

#     data["absorption_detected"][i] = absorption_found
#     data["absorption_relevant"][i] = input(f"{data['sdss_name'][i]}")

In [ ]:
# data["absorption_relevant"] = data["absorption_relevant"].map({"f":False,"t":True,"-":pd.NA})

# # data.to_csv(relevant_erq_data_path)
data = pd.read_csv(relevant_erq_data_path,index_col=0)

data

,sdss_name,z_dr12,rew,fwhm,kt80,specid,redshift,redshift_warning,wavelength,flux,ivar,absorption_detected,absorption_relevant
0,000610.67+121501.2,2.309943,106.996659,4540.061012,0.372288,6360287837779417088,2.309787,0,"[3553.8574198608, 3554.6758199907, 3555.494408...","[1.9252539873, 1.9251964092, 1.3102685213, 0.6...","[0.0, 0.11758627, 0.0, 0.19981963930000002, 0....",False,False
1,000746.19+122223.9,2.429000,219.633801,2432.830081,0.284162,6360267496814303232,2.433351,0,"[3553.8574198608, 3554.6758199907, 3555.494408...","[-5.924967289, -5.9247941971, 3.1096043587, 12...","[0.0, 0.11920769510000001, 0.0, 0.0926503986, ...",False,True
2,002400.25+245031.9,2.809255,143.689957,3523.281699,0.372201,7069533316387723264,2.789841,4,"[3558.7706484879, 3559.5901800608, 3560.409900...","[-0.8786045313, -0.8787579536000001, 0.1475744...","[0.0, 0.0552793145, 0.0, 0.048356846, 0.047023...",False,True
3,004713.21+264024.7,2.563943,100.506344,3685.427543,0.325082,7077626822452860928,2.553878,0,"[3561.2298094276, 3562.0499073084, 3562.870194...","[-1.7795865536000002, -1.7798964977, -1.780206...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",True,True
4,005044.95-021217.6,2.254000,147.225015,4343.057592,0.193256,4920452066097518592,2.253475,0,"[3553.0392081527, 3553.8574198608, 3554.675819...","[1.8903125525000002, 1.8905686140000002, 1.890...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,223754.52+065026.6,2.608809,140.676812,1390.574742,0.356129,5694835917824743424,2.614897,0,"[3561.2298094276, 3562.0499073084, 3562.870194...","[14.607211113, -22.7902507782, 20.6671199799, ...","[0.09246767310000001, 0.1343497932, 0.12748730...",False,True
93,225438.30+232714.5,3.087494,415.244897,4411.577590,0.358143,7417535898320132096,3.087466,0,"[3568.6174928348, 3569.4392919885, 3570.261280...","[-1.3009853363000001, -0.3360394239, -0.240143...","[0.2848583758, 0.26905199890000003, 0.28210353...",False,True
94,232326.17-010033.1,2.356070,255.993179,3989.269919,0.370693,4740064262789289984,2.350765,0,"[3564.5113342624, 3565.3321878293, 3566.153230...","[7.0590491295, -4.7015686035, 7.8777022362, 1....","[0.0343443416, 0.1265157163, 0.126755625, 0.13...",False,False
95,232828.47+044346.8,2.563872,358.595759,1583.693351,0.352179,4822392127110797312,2.568358,0,"[3553.0392081527, 3553.8574198608, 3554.675819...","[1.856464982, 1.8564176559, 1.8563705683, 1.85...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",True,True
